# 04. 최신 프레임과 판단 유지 시간 / 75분

**강사 설명 → 완성된 셀 실행 → 조건 비교 → 결과 해설** 순서입니다. 코드 작성 과제나 관찰 답안 제출은 없습니다.

대기 프레임을 최신 한 장으로 제한하는 것과 이미 나온 판단을 얼마 동안 유지하는 것은 별개입니다.

[단원 실행 안내](README.md) · [슬라이드 실습 시작](https://docs.google.com/presentation/d/1J5Jw6Lg-2oYFKec-RROYlt6nxtG-IHVHd7hLwH8gd9s/edit#slide=id.d4_lab04_start)

각 단원은 새 커널에서 시작합니다. 노트북 셀은 위에서 아래로 실행하고, `bash` 명령은 지정한 터미널에서 실행합니다. 결과가 이미 있으면 재사용 여부와 실행 이름을 확인합니다.


## 1. 아키텍처 — 지금 실행하는 부분

![최신 프레임과 판단 유지 시간의 입력·처리·출력](../교재그림/guided-vla/day4-04.svg)

put은 대기 슬롯보다 오래된 프레임을 거부하고 새 입력을 깊은 복사로 보관합니다. take는 슬롯을 비우면서 프레임 하나를 반환합니다. 추론이 끝나면 이미 새로운 프레임이 기다릴 수 있습니다. 제어 timer는 판단이 갱신될 때까지 같은 issued_ms로 만료 여부를 검사합니다.

**그림을 읽는 순서:** 1→2→3 뒤 아래 4→5→6으로 이어집니다. 실시간 반복은 새 관측이 다음 입력으로 들어오는 과정이며, 저장 결과를 읽는 셀은 모델을 다시 실행하지 않습니다.


### 전체 실행 지도에서 현재 단원 찾기

![4일차 실행 경계 전체 지도](../교재그림/guided-vla/detailed/day4-system.svg)

터미널 A/B/C는 실행·관찰 역할이고 ROS 노드와 HTTP 서비스는 실제 처리 주체입니다. 02는 파일 요청만, 05는 독립 YOLO 실행만 다루므로 매 단원에서 모든 상자를 동시에 실행하지 않습니다. 아래 단원별 도식과 함께 현재 경로를 확인합니다.

## 상세 해설 — 앞선 수업 없이도 흐름을 읽기

아래 설명은 1~3일차에서 배운 개념도 이번 실습의 입력·코드·출력에 맞춰 다시 풉니다. 실행 중에는 필요한 부분을 읽고, 수업 뒤에는 독립적인 참고 자료로 사용할 수 있습니다. 추가 작성 과제는 없습니다.

### 1. 기초 복습: 세 시각과 두 종류의 지연

- captured_ms는 관측의 나이, issued_ms는 승인의 나이입니다.
- pose_ms는 거북이 상태를 마지막으로 받은 시각입니다.
- TTL과 max_age는 서로 다른 시계를 제한합니다.
- monotonic 지연 측정과 ROS 시각 비교를 섞지 않습니다.

captured_ms는 카메라 header에서 얻는 영상의 시각이고 issued_ms는 처리 결과가 만들어진 시각입니다. now-captured가 관측 age, now-issued가 승인 age입니다. 오래 걸린 추론이 방금 끝났다면 승인 age는 작아도 관측 age는 클 수 있습니다. 따라서 TTL만 검사하면 오래된 장면에 대한 새 응답을 오래 사용할 수 있습니다.

pose_ms는 현재 거북이 위치 메시지를 받은 시각입니다. 영상과 모델이 정상이어도 Pose 갱신이 끊기면 올바른 방향 오차를 계산할 수 없어 정지합니다. 이 세 시간 조건은 모두 만족해야 합니다. 코드의 비교는 0 이상이면서 제한보다 작아야 하므로 제한값과 정확히 같으면 만료입니다.

ROS clock의 timestamp는 관측과 제어 시각을 비교하는 기준입니다. time.monotonic은 요청 소요시간을 재는 별도 시계입니다. 서로 기준점이 다른 수치를 직접 빼지 않습니다. 화면에서 표시하는 age와 latency가 비슷한 단어라도 포함하는 대기 구간과 기준 시각을 먼저 확인해야 합니다.


### 2. 실행 아키텍처: 대기 한 장과 실행 중 한 장

![실행 주체와 데이터 경계](../교재그림/guided-vla/detailed/day4-04.svg)

- 카메라 A·B·C·D 중 worker가 A를 처리할 수 있습니다.
- 대기 슬롯은 B → C → D로 교체되어 D만 남습니다.
- A 추론은 취소되지 않으며 완료 후 D를 처리합니다.
- timer는 그 사이에도 현재 승인의 만료를 검사합니다.

FIFO 큐는 먼저 온 입력을 모두 순서대로 처리합니다. 카메라보다 모델이 느리면 큐가 쌓이고 제어에 쓰는 이미지가 점점 과거가 됩니다. 최신 대기 슬롯은 아직 시작하지 않은 입력을 최신 한 장으로 교체해 이 누적을 줄입니다. 이미 시작한 A의 추론까지 D로 바꿔 주는 기능은 아닙니다.

노트북의 LatestFrameMailbox는 put/take와 깊은 복사, 오래된 대기 입력 거부를 분리해서 보여 주는 계약 예제입니다. 실제 HandFollow 노드는 이 클래스를 직접 생성하지 않고 on_image의 self.latest 교체와 tick의 제출 로직으로 같은 대기 한 장 전략을 구현합니다. 따라서 예제 클래스에 넣은 모든 세부 검사가 ROS 슬롯에도 자동 적용된다고 설명하면 안 됩니다.

A가 완료될 때 worker 결과와 대기 D는 서로 다릅니다. 현재 제어는 A의 관측과 A의 판단을 한 묶음으로 쓰고, 다음 worker가 D를 처리합니다. 이 구조를 보면 최신 입력만 남겨도 결과 지연이 0이 되지 않는 이유와 timer가 독립적으로 필요한 이유가 드러납니다.


### 3. 코드 추적: 만료 시각 계산 예시

- 예시 captured=1000, issued=1600, TTL=400ms
- now=1999: 승인 age 399ms, TTL 검사 통과
- now=2000: 승인 age 400ms, expired
- 실제 유효 종료는 TTL·frame age 등 먼저 온 한계

이 숫자는 계산 설명용이며 실제 추론 속도 측정값이 아닙니다. max_age=5000이고 Pose가 충분히 새롭다고 가정하면 captured=1000, issued=1600인 판단은 승인 시각부터 400ms가 되기 직전까지 TTL 조건을 만족합니다. now=2000에서 정확히 400ms이므로 expired가 됩니다. timer는 검사 주기에 이 조건을 평가하므로 벽시계상 정지 관찰 시점은 계산 경계와 완전히 같다고 보장하지 않습니다.

다른 조건을 모두 만족한다고 가정할 때 시간상 상한은 min(issued+ttl, captured+max_age)로 이해할 수 있습니다. 여기에 Pose freshness도 만족해야 하며 새 STOP·도착·오류는 더 일찍 정지시킬 수 있습니다. TTL을 3000으로 늘려도 frame age 제한이 먼저 도달하면 계속 움직이지 않습니다.

put 다음 take를 두 번 호출하는 예제에서 첫 take는 마지막 대기 항목을 돌려주고 두 번째는 빈 상태입니다. 반환된 데이터가 원본 변경에 영향을 받는지 확인하는 깊은 복사 검사는 비동기 처리에서 입력 버퍼가 바뀌는 문제를 이해하기 위한 것입니다.


### 4. TTL 비교를 해석하고 수정하기

- 400·1000·3000ms 비교에서 다른 조건을 유지합니다.
- 긴 TTL은 끊김을 줄여도 새 STOP 반응을 늦출 수 있습니다.
- 최초 모델 적재와 반복 요청 지연을 구분합니다.
- 상한 변경 전에 age·reason·judgment 간격을 확인합니다.

TTL이 짧으면 판단 사이에 정지 구간이 생길 수 있고, 길면 이전 FOLLOW를 더 오래 사용할 수 있습니다. 움직임이 부드러워졌다는 것과 현재 손 상태에 빠르게 반응한다는 것은 다른 성질입니다. 손을 주먹으로 바꾼 뒤 모델이 새 STOP을 내기 전까지 기존 판단이 유효할 수 있음을 함께 설명합니다.

합성 셀의 경계값은 결정적 검사를 보여 주고 실제 로그는 모델·카메라·시스템 부하가 포함된 동작을 보여 줍니다. 최초 모델 적재가 포함된 요청을 반복 추론과 섞어 평균내면 TTL 선택 이유가 흐려집니다. 같은 입력 절차와 max_age를 유지하고 judgment 사이 간격, action의 reason, 이동·정지 구간을 같이 읽습니다.

직접 수정할 부분은 launch의 ttl_ms·max_age_ms와 core의 비교 경계입니다. 대기 전략을 바꾼다면 단순히 큐 크기만 늘리지 말고 버릴 입력, 실행 중 입력, 제어 중 입력의 소유권을 따로 설계해야 합니다. 현재 수업에서는 수강생이 이 로직을 작성하지 않고 제공된 조건별 실행으로 결과를 확인합니다.


## 2. 코드 위치와 변경 범위

| 구분 | 이 단원에서 확인할 내용 |
|---|---|
| 실제 소스 | [ros_ws/src/nepes_vla_demo/nepes_vla_core/contracts.py](ros_ws/src/nepes_vla_demo/nepes_vla_core/contracts.py) |
| 바꾸어 볼 조건 | `ttl_ms:=400 / 1000 / 3000` |
| 예상되는 차이 | 고정 시각 예제에서 TTL 직전과 정확한 경계의 FOLLOW/STOP을 확인합니다. 실제 실행에서는 첫 적재 시간을 분리하고 세 TTL 로그의 이동·정지 구간을 비교합니다. |
| 직접 구현·수정할 때 | 최신 슬롯은 추론 중인 프레임을 취소하지 않습니다. TTL은 issued_ms 기준이고 max_age_ms는 captured_ms 기준입니다. TTL만 늘리면 오래된 판단으로 계속 움직일 수 있으므로 정지 반응도 함께 봅니다. |

설정값 변경은 아래 준비된 비교 실행으로 진행합니다. 소스를 직접 변경하는 경우에는 실행 중인 프로세스를 종료하고 다시 실행해야 합니다. ROS 패키지 파일은 재빌드·환경 source까지 확인합니다.


## 3. 준비 — 실제 실행 코드와 같은 함수를 사용합니다

아래 CPU 예제는 카메라·모델·ROS 없이 자료의 흐름을 분리해 보여 줍니다. 통과가 실제 인식 성공이나 장비 성능을 뜻하지는 않습니다.


In [ ]:
from pathlib import Path
import sys, json, inspect, uuid
from datetime import datetime, timezone
from IPython.display import display, Markdown, Code
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'09.Pseudo-VLA_v2').is_dir())
LESSON = ROOT/'09.Pseudo-VLA_v2'
if str(LESSON) not in sys.path:
    sys.path.insert(0, str(LESSON))
# 매 단원 새 커널: 다른 일차의 common 모듈이 남아 있지 않게 합니다.
RUN = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:8]
OUT = ROOT/'Data/outputs/09.Pseudo-VLA_v2/guided'/RUN

# 각 행은 고정 입력의 한 조건입니다. 합성 예제 통과를 실제 장비의 정확도·속도로 해석하지 않습니다.
def show_rows(rows):
    """조건·결과를 한 표에서 읽습니다. 수강생이 답안 dict를 채우지 않습니다."""
    if not rows:
        print('표시할 결과 없음')
        return
    keys = list(dict.fromkeys(key for row in rows for key in row))
    def value(v):
        return str(v).replace('|', '/').replace('\n', ' ')
    lines = [' | '.join(keys), ' | '.join(['---']*len(keys))]
    lines += [' | '.join(value(row.get(key, '')) for key in keys) for row in rows]
    display(Markdown('\n'.join(lines)))

# 비교 출력을 자동 저장합니다. 수강생이 관찰 답안을 작성하거나 수치를 옮겨 적지 않습니다.
def save_demo(name, rows):
    """합성 예제임을 파일에도 명시합니다. 모델 측정값이나 수강생 답안이 아닙니다."""
    OUT.mkdir(parents=True, exist_ok=True)
    path = OUT/(name+'.json')
    path.write_text(json.dumps({'kind':'synthetic-guided-demo', 'model_measured':False,
                               'rows':rows}, ensure_ascii=False, indent=2)+'\n')
    print('자동 저장:', path)
print('실행 폴더:', LESSON)


### 완성 코드 읽기

아래 셀은 실행에 쓰는 함수 본문을 그대로 표시합니다. **입력 검사 → 조건 분기 → 결과 반환**을 순서대로 읽습니다. 노트북에 복사한 별도 정답과 실제 동작이 어긋나지 않도록 같은 소스를 조회합니다.


In [ ]:
from common.contracts import LatestFrameMailbox
# 함수의 실제 파일과 주석을 함께 표시합니다. 이 셀은 소스를 수정하지 않습니다.
for function in (LatestFrameMailbox,):
    print(inspect.getsourcefile(function))
    display(Code(inspect.getsource(function), language='python'))


### 한 줄씩 읽는 처리 과정

put은 대기 슬롯보다 오래된 프레임을 거부하고 새 입력을 깊은 복사로 보관합니다. take는 슬롯을 비우면서 프레임 하나를 반환합니다. 추론이 끝나면 이미 새로운 프레임이 기다릴 수 있습니다. 제어 timer는 판단이 갱신될 때까지 같은 issued_ms로 만료 여부를 검사합니다.

최신 슬롯은 추론 중인 프레임을 취소하지 않습니다. TTL은 issued_ms 기준이고 max_age_ms는 captured_ms 기준입니다. TTL만 늘리면 오래된 판단으로 계속 움직일 수 있으므로 정지 반응도 함께 봅니다.


## 4. 조건을 바꾸어 실행하고 표로 비교하기

고정 시각 예제에서 TTL 직전과 정확한 경계의 FOLLOW/STOP을 확인합니다. 실제 실행에서는 첫 적재 시간을 분리하고 세 TTL 로그의 이동·정지 구간을 비교합니다.

아래 입력은 경계 조건을 설명하기 위한 고정 예제입니다. 주석을 읽고 셀 전체를 실행하면 비교표와 검사 결과가 자동으로 나옵니다.


In [ ]:
from common.contracts import Frame, LatestFrameMailbox
from common.hand_follow import follow_motion, rule_decision

# FIFO처럼 모두 쌓지 않고 아직 처리하지 않은 최신 프레임 하나만 보관합니다.
mailbox = LatestFrameMailbox()
for index, stamp in enumerate((1000, 1100, 1200)):
    mailbox.put(Frame(f'f{index}', stamp, {'pixels': [index]}))
latest = mailbox.take()  # 소비하면 슬롯이 비워집니다.
print('처리할 frame:', latest.frame_id, '교체 횟수:', mailbox.replaced, '다음 take:', mailbox.take())
assert latest.frame_id == 'f2' and mailbox.replaced == 2

obs = dict(frame_id='ttl-demo', captured_ms=1000, center=[1, .5], state='NON_FIST')
decision = rule_decision(obs)
rows = []
for ttl in (400, 1000, 3000):
    for elapsed in (ttl - 1, ttl):
        # 승인 시각은 고정합니다. pose만 갱신해 TTL 이외의 실패를 배제합니다.
        action = follow_motion(obs, decision, (5.5, 5.5, 0), now_ms=1000+elapsed,
                               issued_ms=1000, pose_ms=1000+elapsed, ttl_ms=ttl)
        rows.append(dict(ttl_ms=ttl, elapsed_ms=elapsed, **action))
# 각 행은 고정 입력의 한 조건입니다. 합성 예제 통과를 실제 장비의 정확도·속도로 해석하지 않습니다.
show_rows(rows)
assert [r['command'] for r in rows] == ['FOLLOW', 'STOP'] * 3
# 비교 출력을 자동 저장합니다. 수강생이 관찰 답안을 작성하거나 수치를 옮겨 적지 않습니다.
save_demo('ttl-boundary', rows)


## 5. 실제 입력 실행 — 화면과 로그 연결

본수업은 로컬 카메라 화면을 함께 보며 역할을 교대합니다. 아래의 A/B/C는 터미널 역할입니다. 카메라와 제어기는 각각 하나만 실행합니다. 실제 실행의 지연과 인식 결과는 위 합성 예제와 별도로 확인합니다.


## 실제 ROS 실행 — JupyterLab 터미널 A·B·C

<!-- procedure:ros-terminals-57bc75:start -->
![새 터미널 준비와 세 역할](../교재그림/절차/ros-terminals.svg)

1–3을 새 터미널마다 적용합니다. 4–6은 동시에 실행하는 역할이며 노드·토픽 이름은 해당 실습 명령을 따릅니다.
<!-- procedure:ros-terminals-57bc75:end -->


새 Terminal에서는 아래 블록의 첫 줄 bash부터 전체를 복사합니다. 로컬·원격 모두 오프라인 수기 서명 명부의 본인 번호를 ROS_DOMAIN_ID로 사용합니다. 예시 7을 본인 번호로 바꾸고 본인 A·B·C 터미널에 같은 값을 적용합니다. 공동 통신 실습에서만 강사가 지정한 동일 명부 번호를 사용합니다. 노트북 Python 셀에 붙이지 않습니다. 이미 환경을 설정한 같은 B에서 모드·TTL만 바꿀 때는 Ctrl+C 후 launch 명령만 실행합니다.

**처음 준비 또는 패키지를 찾지 못할 때 — 빌드:**
```bash
bash
bash /workspace/08.ROS_v2/scripts/build_workspace.sh
bash /workspace/09.Pseudo-VLA_v2/scripts/build_workspace.sh
```

**A — 카메라 입력:** 이미 A가 실행 중이면 그대로 유지합니다. 카메라 publisher는 하나만 실행합니다.
```bash
bash
source /opt/ros/humble/setup.bash
source /workspace/Data/build/08.ROS_v2/install/setup.bash
source /workspace/Data/build/09.Pseudo-VLA_v2/install/setup.bash
export ROS_DOMAIN_ID=7 # 7을 본인 수기 서명 명부 번호로 변경
ros2 run v4l2_camera v4l2_camera_node --ros-args -p video_device:=/dev/video0 -p image_size:="[640, 480]" -r image_raw:=/camera/image_raw
```

카메라 번호는 배정값입니다. 녹화 입력으로 비교할 때는 A를 Ctrl+C로 종료하고 같은 A에서 아래 명령을 실행합니다. 매 조건에서 같은 시작 지점으로 재생을 다시 시작합니다.
```bash
ros2 run nepes_ros_lab recorded_image_publisher --ros-args -p image_source:=/workspace/Data/videos/07.Mediapipe/person_open_palm.avi
```

**화면 읽기:** /vla/image_overlay의 초록색 점은 전진·회전 중, 빨간색 점은 좌표가 있지만 정지 중입니다. 좌표가 없으면 NO TARGET입니다. Target은 거북이 목표 좌표, age는 제어에 사용한 프레임 경과 시간입니다. 승인 경과 시간·TTL·남은 유효시간도 확인합니다. VLM 추론 중에는 이전 관측 화면이 유지됩니다. 최신 원본은 /camera/image_raw입니다. mirror:=true는 목표 x만 반전하고 영상과 점은 원본 위치를 유지합니다.

**B — 첫 TTL 조건:**
```bash
bash
source /opt/ros/humble/setup.bash
source /workspace/Data/build/08.ROS_v2/install/setup.bash
source /workspace/Data/build/09.Pseudo-VLA_v2/install/setup.bash
export ROS_DOMAIN_ID=7 # 7을 본인 수기 서명 명부 번호로 변경
ros2 launch nepes_vla_demo hand_follow.launch.py mode:=vlm \
  ttl_ms:=400 \
  vlm_model:=gemma4:e2b-it-qat \
  log_path:=/workspace/Data/logs/09.Pseudo-VLA_v2/04-ttl-0400.jsonl
```

앞 조건을 Ctrl+C로 종료한 **같은 B에서** 실행합니다. turtlesim을 새로 시작해 초기 pose를 맞추고 녹화 입력도 같은 시작 지점으로 맞춥니다.
```bash
ros2 launch nepes_vla_demo hand_follow.launch.py mode:=vlm \
  ttl_ms:=1000 \
  vlm_model:=gemma4:e2b-it-qat \
  log_path:=/workspace/Data/logs/09.Pseudo-VLA_v2/04-ttl-1000.jsonl
```

앞 조건을 Ctrl+C로 종료한 **같은 B에서** 실행합니다. turtlesim을 새로 시작해 초기 pose를 맞추고 녹화 입력도 같은 시작 지점으로 맞춥니다.
```bash
ros2 launch nepes_vla_demo hand_follow.launch.py mode:=vlm \
  ttl_ms:=3000 \
  vlm_model:=gemma4:e2b-it-qat \
  log_path:=/workspace/Data/logs/09.Pseudo-VLA_v2/04-ttl-3000.jsonl
```

**C — 관측·판단·행동 확인:** 새 Terminal에서 전체를 실행합니다.
```bash
bash
source /opt/ros/humble/setup.bash
source /workspace/Data/build/08.ROS_v2/install/setup.bash
source /workspace/Data/build/09.Pseudo-VLA_v2/install/setup.bash
export ROS_DOMAIN_ID=7 # 7을 본인 수기 서명 명부 번호로 변경
ros2 topic echo /vla/action
```

연속 출력은 Ctrl+C로 끝낸 뒤, 같은 C에서 필요에 따라 ros2 topic echo /vla/perception 또는 ros2 topic echo /vla/decision을 실행합니다. frame_id·reason·TTL·승인 경과 시간을 영상과 비교합니다.

**종료:** A 중단 → C에서 STOP 확인 → B 종료 → C 종료. 다음 노트북으로 바로 이어갈 때는 A를 유지하고 B만 종료합니다. GUI 없는 자습은 B launch 앞에 QT_QPA_PLATFORM=offscreen, 인자에 with_image_view:=false를 사용합니다.


## 완료 검사 — 같은 입력으로 3조건 비교

각 조건을 3회 반복하고 모델 적재 직후 첫 판단은 별도 기록합니다.

1. 중앙에서 떨어진 손 목표로 FOLLOW를 받아 이동 구간을 확보합니다.
2. 주먹 전환·손 제거·다른 위치 이동을 각각 따로 수행합니다. 원본 장면의 전환 시각을 관찰자가 기록합니다.
3. STOP 시각·이유를 기록합니다. 목표 도착과 TTL 만료를 구분합니다.
4. 이전 목표로 더 움직인 시간, 정지 구간, 전환 후 정지 반응을 비교합니다. 과거 관측 화면만으로 실제 손 전환 시각을 정하지 않습니다.
5. 다음 판단이 없을 때의 만료 경계는 아래 고정 입력 셀로 검증합니다.

| TTL(ms) | 반복 | 판단 지연 | 정지 구간 | 주먹·손 제거 후 정지 지연 | 이전 목표로 더 움직임 | STOP 이유 |
|---:|---:|---|---|---|---|---|
| 400 | 1–3 | 미측정 | 미측정 | 미측정 | 미측정 | 미측정 |
| 1000 | 1–3 | 미측정 | 미측정 | 미측정 | 미측정 | 미측정 |
| 3000 | 1–3 | 미측정 | 미측정 | 미측정 | 미측정 | 미측정 |

로그는 04-ttl-0400.jsonl, 04-ttl-1000.jsonl, 04-ttl-3000.jsonl입니다. 추가 반복은 log_path에 회차 이름을 붙여 구분합니다. 전환 시각을 기록하지 않았다면 반응 시간은 미측정입니다.


## 로그 요약과 결과 저장

파일마다 마지막 session만 요약합니다. 첫 판단 지연은 이후 판단과 분리합니다. 행동 기록 사이의 시간을 집계하므로 마지막 로그 이후는 포함하지 않습니다. 정답률·손 전환 후 반응은 자동 추정하지 않습니다. 아래 셀은 실제 결과가 없으면 미측정으로 저장합니다.


In [ ]:
from datetime import datetime, timezone
import uuid
from common.latency_report import summarize_log
reports = []
for ttl in (400, 1000, 3000):
    path = ROOT / "Data/logs/09.Pseudo-VLA_v2" / f"04-ttl-{ttl:04d}.jsonl"
    report = summarize_log(path, expected_ttl_ms=ttl)
    reports.append(report)
    print(json.dumps(report, ensure_ascii=False, indent=2))
run = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8]
out = ROOT / "Data/outputs/09.Pseudo-VLA_v2/latency" / run
out.mkdir(parents=True, exist_ok=False)
(out / "summary.json").write_text(json.dumps(reports, ensure_ascii=False, indent=2)+"\n")
print("Saved:", out / "summary.json")


## 6. 결과 해설과 다음 단계

대기 프레임을 최신 한 장으로 제한하는 것과 이미 나온 판단을 얼마 동안 유지하는 것은 별개입니다.

고정 시각 예제에서 TTL 직전과 정확한 경계의 FOLLOW/STOP을 확인합니다. 실제 실행에서는 첫 적재 시간을 분리하고 세 TTL 로그의 이동·정지 구간을 비교합니다.

이미지 의미 판단의 비용을 확인했으므로 다음에는 빠른 검출기가 좌표를 추출하는 구조를 봅니다.

비교표와 실제 화면·로그를 강사 설명과 대조합니다. 별도 답안 작성은 없으며, 자동 저장 파일과 실행 결과로 진행 상태를 확인합니다.
